In [8]:
import pandas as pd
import numpy as np

In [9]:
df_heart = pd.read_csv('/Users/sukitharathnayake/CodeRepo/ST4035/Data/ST 4035 Heart Data.csv', index_col=0)

In [10]:
df_heart.head()

,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
1,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
2,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
3,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
4,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
5,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


In [11]:
df_heart.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303 entries, 1 to 303
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        303 non-null    int64  
 1   Sex        303 non-null    int64  
 2   ChestPain  303 non-null    object 
 3   RestBP     303 non-null    int64  
 4   Chol       303 non-null    int64  
 5   Fbs        303 non-null    int64  
 6   RestECG    303 non-null    int64  
 7   MaxHR      303 non-null    int64  
 8   ExAng      303 non-null    int64  
 9   Oldpeak    303 non-null    float64
 10  Slope      303 non-null    int64  
 11  Ca         299 non-null    float64
 12  Thal       301 non-null    object 
 13  AHD        303 non-null    object 
dtypes: float64(2), int64(9), object(3)
memory usage: 35.5+ KB


In [12]:
num_cols = df_heart.select_dtypes(include=['int64','float64']).columns
cat_cols = df_heart.select_dtypes(include=['object','category']).columns

df_heart[num_cols] = df_heart[num_cols].fillna(df_heart[num_cols].median())
df_heart[cat_cols] = df_heart[cat_cols].fillna(df_heart[cat_cols].mode().iloc[0])

In [13]:
df_heart.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303 entries, 1 to 303
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        303 non-null    int64  
 1   Sex        303 non-null    int64  
 2   ChestPain  303 non-null    object 
 3   RestBP     303 non-null    int64  
 4   Chol       303 non-null    int64  
 5   Fbs        303 non-null    int64  
 6   RestECG    303 non-null    int64  
 7   MaxHR      303 non-null    int64  
 8   ExAng      303 non-null    int64  
 9   Oldpeak    303 non-null    float64
 10  Slope      303 non-null    int64  
 11  Ca         303 non-null    float64
 12  Thal       303 non-null    object 
 13  AHD        303 non-null    object 
dtypes: float64(2), int64(9), object(3)
memory usage: 35.5+ KB


In [14]:
from sklearn.model_selection import train_test_split

X = df_heart.drop('AHD', axis=1)
y = df_heart['AHD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# --- Encoding ---
# One-hot encode categorical features: ChestPain, Thal
X_train_enc = pd.get_dummies(X_train, columns=['ChestPain', 'Thal'], drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=['ChestPain', 'Thal'], drop_first=True)

# Align columns — ensures test set has same dummy columns as train set
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# Label-encode the target: Yes -> 1, No -> 0
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print('Encoded feature shape:', X_train_enc.shape)
print('Target classes:', le.classes_)

Encoded feature shape: (242, 16)
Target classes: ['No' 'Yes']


In [16]:
# --- Feature Scaling (StandardScaler) ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)  # fit on train only
X_test_scaled  = scaler.transform(X_test_enc)       # transform test

print('Scaling complete. Each feature now has mean ≈ 0 and std ≈ 1 on the training set.')

Scaling complete. Each feature now has mean ≈ 0 and std ≈ 1 on the training set.


In [17]:
# --- Logistic Regression ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_enc)

y_pred = lr.predict(X_test_scaled)

print(f'Accuracy : {accuracy_score(y_test_enc, y_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print('Confusion Matrix:')
print(confusion_matrix(y_test_enc, y_pred))

Accuracy : 0.8689

Classification Report:
              precision    recall  f1-score   support

          No       0.89      0.83      0.86        29
         Yes       0.85      0.91      0.88        32

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.87      0.87      0.87        61

Confusion Matrix:
[[24  5]
 [ 3 29]]
